# Step 5: Model Evaluation - YOGA Chatbot

**Chatbot Name**: YOGA (YOgyakarta Guide Assistant)

**Objective**: Comprehensive evaluation of trained LSTM model

**Date**: 2025-11-26

**Input**: Trained LSTM model from Step 4

**Evaluation Steps**:
1. Load trained model and test data
2. Overall performance metrics
3. Per-class performance analysis
4. Confusion matrix analysis
5. Error analysis
6. Prediction examples
7. Model inference test

**Model**: Bidirectional LSTM for 92-class intent classification

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, cohen_kappa_score
)

# NLP preprocessing
from gensim.models import Word2Vec
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow version: {tf.__version__}")
print("✓ Libraries imported successfully")

## 2. Load Model and Data

In [ ]:
print("=" * 80)
print("LOADING MODEL AND DATA")
print("=" * 80)

# Load best model
model_path = '../models/yoga_lstm_best.h5'
model = load_model(model_path)
print(f"✓ Loaded model: {model_path}")

# Load model info
with open('../models/model_info.json', 'r') as f:
    model_info = json.load(f)
print(f"✓ Loaded model info")

# Load test data
X = np.load('../data/features/X.npy')
y = np.load('../data/features/y.npy')
print(f"✓ Loaded feature data: X{X.shape}, y{y.shape}")

# Load label encoder
with open('../data/features/label_encoder.pickle', 'rb') as f:
    label_encoder = pickle.load(f)
print(f"✓ Loaded label encoder: {len(label_encoder.classes_)} classes")

# Load Word2Vec model
w2v_model = Word2Vec.load('../data/features/word2vec.model')
print(f"✓ Loaded Word2Vec model")

# Load feature extraction info
with open('../data/features/feature_extraction_info.json', 'r', encoding='utf-8') as f:
    feature_info = json.load(f)
print(f"✓ Loaded feature extraction info")

# Split data (same as training)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\n✓ Test set: {len(X_test)} samples")
print(f"✓ Unique classes in test: {len(np.unique(y_test))}")

## 3. Model Summary

In [ ]:
print("=" * 80)
print("MODEL SUMMARY")
print("=" * 80)

print(f"\nModel Information:")
print(f"  Name: {model_info['model_name']}")
print(f"  Architecture: {model_info['architecture']}")
print(f"  Total parameters: {model_info['total_params']:,}")
print(f"  Input shape: {model_info['input_shape']}")
print(f"  Output classes: {model_info['num_classes']}")

print(f"\nTraining Information:")
print(f"  Epochs trained: {model_info['epochs_trained']} / {model_info['max_epochs']}")
print(f"  Batch size: {model_info['batch_size']}")
print(f"  Optimizer: {model_info['optimizer']} (lr={model_info['learning_rate']})")
print(f"  Loss function: {model_info['loss']}")

print(f"\nRecorded Performance:")
print(f"  Test accuracy: {model_info['test_accuracy']*100:.2f}%")
print(f"  Test loss: {model_info['test_loss']:.4f}")

model.summary()

## 4. Overall Performance Metrics

In [ ]:
print("=" * 80)
print("OVERALL PERFORMANCE EVALUATION")
print("=" * 80)

# Make predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

# Per-class metrics
all_labels = list(range(len(label_encoder.classes_)))
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=all_labels, zero_division=0
)

# Macro and weighted averages
macro_precision = np.mean(precision)
macro_recall = np.mean(recall)
macro_f1 = np.mean(f1)

# Weighted by support
total_support = support.sum()
weighted_precision = np.sum(precision * support) / total_support
weighted_recall = np.sum(recall * support) / total_support
weighted_f1 = np.sum(f1 * support) / total_support

print(f"\n📊 Overall Metrics:")
print(f"  Accuracy: {accuracy*100:.2f}%")
print(f"  Cohen's Kappa: {kappa:.4f}")

print(f"\n📊 Macro Average (unweighted):")
print(f"  Precision: {macro_precision*100:.2f}%")
print(f"  Recall: {macro_recall*100:.2f}%")
print(f"  F1-Score: {macro_f1*100:.2f}%")

print(f"\n📊 Weighted Average (by support):")
print(f"  Precision: {weighted_precision*100:.2f}%")
print(f"  Recall: {weighted_recall*100:.2f}%")
print(f"  F1-Score: {weighted_f1*100:.2f}%")

print(f"\n📊 Test Set Statistics:")
print(f"  Total samples: {len(y_test)}")
print(f"  Classes present: {len(np.unique(y_test))} / {len(label_encoder.classes_)}")
print(f"  Classes missing: {len(label_encoder.classes_) - len(np.unique(y_test))}")

## 5. Per-Class Performance Analysis

In [ ]:
print("=" * 80)
print("PER-CLASS PERFORMANCE")
print("=" * 80)

# Create detailed metrics dataframe
metrics_df = pd.DataFrame({
    'Class': label_encoder.classes_,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

# Add accuracy per class
class_accuracy = []
for i in range(len(label_encoder.classes_)):
    mask = y_test == i
    if mask.sum() > 0:
        acc = (y_pred[mask] == i).sum() / mask.sum()
    else:
        acc = 0.0
    class_accuracy.append(acc)

metrics_df['Accuracy'] = class_accuracy

# Filter classes present in test set
metrics_present = metrics_df[metrics_df['Support'] > 0].copy()
metrics_missing = metrics_df[metrics_df['Support'] == 0].copy()

print(f"\n📊 Classes in test set: {len(metrics_present)}")
print(f"📊 Classes not in test set: {len(metrics_missing)}")

# Sort by F1-score
metrics_present_sorted = metrics_present.sort_values('F1-Score', ascending=False)

print(f"\n🏆 Top 10 Best Performing Classes:")
print(metrics_present_sorted.head(10)[['Class', 'Precision', 'Recall', 'F1-Score', 'Support']].to_string(index=False))

print(f"\n⚠️  Bottom 10 Worst Performing Classes:")
print(metrics_present_sorted.tail(10)[['Class', 'Precision', 'Recall', 'F1-Score', 'Support']].to_string(index=False))

# Save full metrics
metrics_df.to_csv('../models/evaluation_metrics.csv', index=False)
print(f"\n✓ Saved: ../models/evaluation_metrics.csv")

In [ ]:
# Visualize F1-scores distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# F1-score histogram
ax1.hist(metrics_present['F1-Score'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(macro_f1, color='red', linestyle='--', linewidth=2, label=f'Macro Avg: {macro_f1:.3f}')
ax1.axvline(weighted_f1, color='green', linestyle='--', linewidth=2, label=f'Weighted Avg: {weighted_f1:.3f}')
ax1.set_xlabel('F1-Score', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Classes', fontsize=12, fontweight='bold')
ax1.set_title('F1-Score Distribution Across Classes', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# F1-score by support
scatter = ax2.scatter(metrics_present['Support'], metrics_present['F1-Score'], 
                     c=metrics_present['F1-Score'], cmap='RdYlGn', s=100, alpha=0.6, edgecolors='black')
ax2.set_xlabel('Support (Number of Samples)', fontsize=12, fontweight='bold')
ax2.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax2.set_title('F1-Score vs Support', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax2, label='F1-Score')

plt.tight_layout()
plt.savefig('../models/f1_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: ../models/f1_distribution.png")

## 6. Confusion Matrix Analysis

In [ ]:
print("=" * 80)
print("CONFUSION MATRIX ANALYSIS")
print("=" * 80)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Overall statistics
total_correct = np.trace(cm)
total_predictions = cm.sum()
total_errors = total_predictions - total_correct

print(f"\nConfusion Matrix Statistics:")
print(f"  Total predictions: {total_predictions}")
print(f"  Correct predictions: {total_correct} ({total_correct/total_predictions*100:.2f}%)")
print(f"  Incorrect predictions: {total_errors} ({total_errors/total_predictions*100:.2f}%)")

# Find most confused class pairs
# Set diagonal to 0 to find off-diagonal max
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)

# Get top 10 most confused pairs
confused_pairs = []
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            confused_pairs.append({
                'True': label_encoder.classes_[i],
                'Predicted': label_encoder.classes_[j],
                'Count': cm[i, j]
            })

confused_df = pd.DataFrame(confused_pairs).sort_values('Count', ascending=False)

print(f"\n⚠️  Top 10 Most Confused Class Pairs:")
print(confused_df.head(10).to_string(index=False))

# Save confusion pairs
confused_df.to_csv('../models/confusion_pairs.csv', index=False)
print(f"\n✓ Saved: ../models/confusion_pairs.csv")

## 7. Error Analysis

In [ ]:
print("=" * 80)
print("ERROR ANALYSIS")
print("=" * 80)

# Find misclassified samples
errors_mask = y_test != y_pred
error_indices = np.where(errors_mask)[0]

print(f"\nTotal errors: {len(error_indices)} / {len(y_test)} ({len(error_indices)/len(y_test)*100:.2f}%)")

# Analyze error distribution
error_true_classes = y_test[errors_mask]
error_pred_classes = y_pred[errors_mask]

# Classes with most errors
error_by_class = pd.Series(error_true_classes).value_counts()
error_class_names = [label_encoder.classes_[i] for i in error_by_class.index[:10]]
error_counts = error_by_class.values[:10]

print(f"\n⚠️  Classes with Most Errors:")
for name, count in zip(error_class_names, error_counts):
    print(f"  {name}: {count} errors")

# Confidence analysis for errors
error_confidences = y_pred_prob[errors_mask].max(axis=1)
correct_confidences = y_pred_prob[~errors_mask].max(axis=1)

print(f"\n📊 Prediction Confidence:")
print(f"  Correct predictions - Avg confidence: {correct_confidences.mean():.4f} (±{correct_confidences.std():.4f})")
print(f"  Wrong predictions - Avg confidence: {error_confidences.mean():.4f} (±{error_confidences.std():.4f})")

In [ ]:
# Visualize confidence distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confidence histogram
ax1.hist(correct_confidences, bins=30, alpha=0.7, label='Correct', color='green', edgecolor='black')
ax1.hist(error_confidences, bins=30, alpha=0.7, label='Wrong', color='red', edgecolor='black')
ax1.set_xlabel('Prediction Confidence', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot
ax2.boxplot([correct_confidences, error_confidences], 
            labels=['Correct', 'Wrong'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
ax2.set_ylabel('Prediction Confidence', fontsize=12, fontweight='bold')
ax2.set_title('Confidence Comparison', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../models/confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: ../models/confidence_analysis.png")

## 8. Classification Report

In [ ]:
print("=" * 80)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 80)

report = classification_report(
    y_test,
    y_pred,
    labels=all_labels,
    target_names=label_encoder.classes_,
    zero_division=0,
    digits=4
)

print(report)

# Save report
with open('../models/detailed_classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print("\n✓ Saved: ../models/detailed_classification_report.txt")

## 9. Prediction Examples

In [ ]:
print("=" * 80)
print("SAMPLE PREDICTIONS")
print("=" * 80)

# Show some correct predictions
correct_mask = y_test == y_pred
correct_indices = np.where(correct_mask)[0][:10]

print("✅ Sample Correct Predictions:")
for i, idx in enumerate(correct_indices, 1):
    true_label = label_encoder.classes_[y_test[idx]]
    pred_label = label_encoder.classes_[y_pred[idx]]
    confidence = y_pred_prob[idx].max()
    
    # Get top 3 predictions
    top_3_idx = np.argsort(y_pred_prob[idx])[-3:][::-1]
    top_3 = [(label_encoder.classes_[i], y_pred_prob[idx][i]) for i in top_3_idx]
    
    print(f"{i}. True Label: {true_label}")
    print(f"   Predicted: {pred_label} (confidence: {confidence:.4f})")
    print(f"   Top 3:")
    for j, (label, prob) in enumerate(top_3, 1):
        print(f"     {j}. {label}: {prob:.4f}")
    print()

# Show some wrong predictions
if len(error_indices) > 0:
    wrong_indices = error_indices[:10]
    
    print(" ❌ Sample Wrong Predictions: ")
    for i, idx in enumerate(wrong_indices, 1):
        true_label = label_encoder.classes_[y_test[idx]]
        pred_label = label_encoder.classes_[y_pred[idx]]
        confidence = y_pred_prob[idx].max()
        
        # Get top 3 predictions
        top_3_idx = np.argsort(y_pred_prob[idx])[-3:][::-1]
        top_3 = [(label_encoder.classes_[i], y_pred_prob[idx][i]) for i in top_3_idx]
        
        print(f"{i}. True Label: {true_label}")
        print(f"   Predicted: {pred_label} (confidence: {confidence:.4f})")
        print(f"   Top 3:")
        for j, (label, prob) in enumerate(top_3, 1):
            marker = "✓" if label == true_label else " "
            print(f"     {j}. {label}: {prob:.4f} {marker}")
        print()

SyntaxError: unterminated string literal (detected at line 9) (323025850.py, line 9)

## 10. Model Inference Test

In [ ]:
print("=" * 80)
print("INFERENCE TEST")
print("=" * 80)

# Initialize preprocessing tools
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def preprocess_text(text):
    """Preprocess input text"""
    # Lowercase
    text = text.lower()
    # Remove stopwords
    text = stopword_remover.remove(text)
    # Stem
    text = stemmer.stem(text)
    # Tokenize
    tokens = text.split()
    return tokens

def get_sequence_vector(tokens, w2v_model, max_len, vec_size):
    """Convert tokens to sequence vector"""
    seq_vectors = []
    for word in tokens[:max_len]:
        if word in w2v_model.wv:
            seq_vectors.append(w2v_model.wv[word])
        else:
            seq_vectors.append(np.zeros(vec_size))
    
    # Pad
    while len(seq_vectors) < max_len:
        seq_vectors.append(np.zeros(vec_size))
    
    return np.array(seq_vectors)

def predict_intent(text):
    """Predict intent for input text"""
    # Preprocess
    tokens = preprocess_text(text)
    
    # Get sequence vector
    max_len = feature_info['max_length']
    vec_size = feature_info['vector_size']
    seq_vec = get_sequence_vector(tokens, w2v_model, max_len, vec_size)
    
    # Reshape for model
    seq_vec = seq_vec.reshape(1, max_len, vec_size)
    
    # Predict
    pred_prob = model.predict(seq_vec, verbose=0)[0]
    pred_class = np.argmax(pred_prob)
    confidence = pred_prob[pred_class]
    
    # Get label
    intent = label_encoder.classes_[pred_class]
    
    # Get top 3 predictions
    top_3_idx = np.argsort(pred_prob)[-3:][::-1]
    top_3 = [(label_encoder.classes_[i], pred_prob[i]) for i in top_3_idx]
    
    return intent, confidence, top_3

# Test with sample inputs
test_inputs = [
    "halo",
    "selamat pagi",
    "terima kasih",
    "destinasi wisata di yogyakarta",
    "tempat wisata sleman",
    "cuaca hari ini"
]

print("\n🔮 Testing Model Inference:\n")
for text in test_inputs:
    intent, conf, top_3 = predict_intent(text)
    print(f"Input: '{text}'")
    print(f"Predicted Intent: {intent}")
    print(f"Confidence: {conf:.4f}")
    print(f"Top 3 predictions:")
    for i, (label, prob) in enumerate(top_3, 1):
        print(f"  {i}. {label}: {prob:.4f}")
    print()

## 11. Summary Report

In [ ]:
print("=" * 80)
print("EVALUATION SUMMARY REPORT")
print("=" * 80)

summary = {
    'model_name': model_info['model_name'],
    'evaluation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'test_samples': int(len(y_test)),
    'num_classes': len(label_encoder.classes_),
    'classes_in_test': int(len(np.unique(y_test))),
    'overall_metrics': {
        'accuracy': float(accuracy),
        'cohen_kappa': float(kappa),
        'macro_precision': float(macro_precision),
        'macro_recall': float(macro_recall),
        'macro_f1': float(macro_f1),
        'weighted_precision': float(weighted_precision),
        'weighted_recall': float(weighted_recall),
        'weighted_f1': float(weighted_f1)
    },
    'error_analysis': {
        'total_errors': int(len(error_indices)),
        'error_rate': float(len(error_indices) / len(y_test)),
        'avg_confidence_correct': float(correct_confidences.mean()),
        'avg_confidence_wrong': float(error_confidences.mean())
    },
    'best_classes': metrics_present_sorted.head(5)[['Class', 'F1-Score']].to_dict('records'),
    'worst_classes': metrics_present_sorted.tail(5)[['Class', 'F1-Score']].to_dict('records')
}

# Save summary
with open('../models/evaluation_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n📊 Evaluation Summary:")
print(f"\nModel: {summary['model_name']}")
print(f"Evaluated: {summary['evaluation_date']}")
print(f"Test Samples: {summary['test_samples']}")
print(f"\nOverall Performance:")
print(f"  Accuracy: {summary['overall_metrics']['accuracy']*100:.2f}%")
print(f"  Weighted F1: {summary['overall_metrics']['weighted_f1']*100:.2f}%")
print(f"  Cohen's Kappa: {summary['overall_metrics']['cohen_kappa']:.4f}")
print(f"\nError Rate: {summary['error_analysis']['error_rate']*100:.2f}%")

print("\n✓ Saved: ../models/evaluation_summary.json")

print("\n" + "=" * 80)
print("✅ EVALUATION COMPLETE")
print("=" * 80)